# 애니메이션 CBF 추천 시스템

genres, tags, directors, production_companies를 벡터화하여 Cosine Similarity 기반 Top-10 유사 애니메이션을 계산하고 `anime_similar` 테이블에 저장합니다.

## 동일 시리즈 중복 제거 전략
1. **series_id 기반**: 기존 series_id가 있는 애니메이션은 그대로 활용
2. **제목 패턴 기반**: series_id가 null인 경우 제목 정규화 + 제작사로 그룹핑
   - `(자막)/(더빙)` → 동일 작품
   - `시즌N / Part N / N기` → 동일 시리즈
3. **후보군 필터링**: 동일 그룹 내 대표작 하나만 남김 (similarity 최고 → view_count+review_count 최고)
4. 결과를 `series_mapping` 테이블에 저장

In [ ]:
import dotenv
import os
import re
import json
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

In [18]:
dotenv.load_dotenv(dotenv.find_dotenv())

pg = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)
cur = pg.cursor()

In [ ]:
# anime + anime_metadata 조회 (name, production, view/review count 추가)
cur.execute("""
    SELECT a.id, a.name, a.series_id, a.production, a.view_count, a.review_count,
           m.genres, m.tags, m.directors, m.production_companies
    FROM anime a
    LEFT JOIN anime_metadata m ON a.id = m.anime_id
    ORDER BY a.id
""")
rows = cur.fetchall()
print(f"Total anime: {len(rows)}")

In [ ]:
# 데이터 파싱
anime_ids = []
names = []
series_ids = []
productions = []
popularities = []  # view_count + review_count
genres_list = []
tags_list = []
directors_list = []
companies_list = []

for row in rows:
    aid, name, sid, production, view_count, review_count, genres_json, tags_json, directors_json, companies_json = row
    anime_ids.append(aid)
    names.append(name or "")
    series_ids.append(sid)
    productions.append(production or "")
    popularities.append((view_count or 0) + (review_count or 0))

    # genres
    genres = genres_json if isinstance(genres_json, list) else (json.loads(genres_json) if genres_json else [])
    genres_list.append(genres)

    # tags
    tags = tags_json if isinstance(tags_json, list) else (json.loads(tags_json) if tags_json else [])
    tags_list.append(tags)

    # directors -> name만 추출
    directors_raw = directors_json if isinstance(directors_json, list) else (json.loads(directors_json) if directors_json else [])
    directors_list.append([d["name"] for d in directors_raw if isinstance(d, dict) and "name" in d])

    # production_companies -> name만 추출
    companies_raw = companies_json if isinstance(companies_json, list) else (json.loads(companies_json) if companies_json else [])
    companies_list.append([c["name"] for c in companies_raw if isinstance(c, dict) and "name" in c])

print(f"Parsed {len(anime_ids)} anime")

In [ ]:
def normalize_title(name):
    """제목을 정규화하여 동일 시리즈/버전을 그룹핑할 수 있는 base title 추출"""
    s = name.strip()

    # 1) 버전 마커 제거: (자막), (더빙), (무삭제), (무삭제판), (자막판), (더빙판)
    s = re.sub(r'\s*[\(（](자막|더빙|무삭제|무삭제판|자막판|더빙판|한국어 더빙)[\)）]', '', s)

    # 2) 시즌/파트 마커 제거
    #    시즌 2, Season 3, 2기, 3기, 2nd Season, Part 2, part.3, 2쿨, 제2기 등
    s = re.sub(r'\s*(시즌\s*\d+|Season\s*\d+|\d+(st|nd|rd|th)\s*Season)', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s*(제?\d+기|파트\s*\d+|Part\s*\.?\s*\d+|\d+쿨)', '', s, flags=re.IGNORECASE)

    # 3) 로마 숫자 시즌 제거: Title II, Title III, Title IV (단, 제목 끝에 올 때만)
    s = re.sub(r'\s+(II|III|IV|V|VI|VII|VIII|IX|X)$', '', s)

    # 4) 끝에 붙는 숫자 제거: "타이틀 2", "타이틀 3" (하지만 한 글자 숫자만)
    s = re.sub(r'\s+\d{1,2}$', '', s)

    # 5) 공백 정규화
    s = re.sub(r'\s+', ' ', s).strip()

    return s


# --- series_id 기반 그룹 + 제목 패턴 기반 그룹 통합 ---
# group_id: 각 애니메이션이 속한 그룹 (동일 그룹 = 동일 시리즈/작품)
group_ids = [None] * len(anime_ids)
next_group_id = 1

# Phase 1: series_id가 있는 애니메이션 그룹핑
series_to_group = {}
for i, sid in enumerate(series_ids):
    if sid is not None:
        if sid not in series_to_group:
            series_to_group[sid] = next_group_id
            next_group_id += 1
        group_ids[i] = series_to_group[sid]

# Phase 2: series_id가 없는 애니메이션 → 제목 패턴 + 제작사 기반 그룹핑
title_prod_to_group = {}
for i in range(len(anime_ids)):
    if group_ids[i] is not None:
        continue  # 이미 series_id로 그룹 할당됨

    base_title = normalize_title(names[i])
    key = (base_title.lower(), productions[i].lower())

    if key not in title_prod_to_group:
        # 같은 base_title인데 이미 series_id 그룹에 속한 애니가 있는지 확인
        matched_group = None
        for j in range(len(anime_ids)):
            if group_ids[j] is not None and normalize_title(names[j]).lower() == base_title.lower() and productions[j].lower() == productions[i].lower():
                matched_group = group_ids[j]
                break

        if matched_group is not None:
            title_prod_to_group[key] = matched_group
        else:
            title_prod_to_group[key] = next_group_id
            next_group_id += 1

    group_ids[i] = title_prod_to_group[key]

# 그룹 통계
groups_with_multiple = sum(1 for g in set(group_ids) if group_ids.count(g) > 1)
ungrouped = sum(1 for g in group_ids if g is None)
print(f"Total groups: {next_group_id - 1}")
print(f"Groups with 2+ members: {groups_with_multiple}")
print(f"Ungrouped anime: {ungrouped}")

# 그룹핑 결과 샘플 확인 (2개 이상 멤버가 있는 그룹)
from collections import Counter
group_counts = Counter(group_ids)
multi_groups = [g for g, c in group_counts.items() if c >= 2]
for gid in multi_groups[:10]:
    members = [names[i] for i in range(len(anime_ids)) if group_ids[i] == gid]
    print(f"  Group {gid}: {members}")

In [ ]:
# series_mapping 테이블 생성 + 데이터 저장
cur.execute("""
    CREATE TABLE IF NOT EXISTS series_mapping (
        id BIGSERIAL PRIMARY KEY,
        group_name VARCHAR(255) NOT NULL,
        anime_id BIGINT NOT NULL REFERENCES anime(id),
        UNIQUE(anime_id)
    );
""")
cur.execute("DELETE FROM series_mapping;")
pg.commit()

# group_id → group_name 매핑 (정규화된 제목 사용)
group_to_name = {}
for i, gid in enumerate(group_ids):
    if gid not in group_to_name:
        group_to_name[gid] = normalize_title(names[i])

mapping_rows = []
for i, gid in enumerate(group_ids):
    if gid is not None:
        mapping_rows.append((group_to_name[gid], anime_ids[i]))

execute_batch(cur, """
    INSERT INTO series_mapping (group_name, anime_id)
    VALUES (%s, %s)
    ON CONFLICT (anime_id) DO UPDATE SET group_name = EXCLUDED.group_name;
""", mapping_rows, page_size=1000)

pg.commit()
print(f"series_mapping: {len(mapping_rows)} rows inserted")

In [21]:
# MultiLabelBinarizer로 One-hot 벡터화
mlb_genres = MultiLabelBinarizer()
mlb_tags = MultiLabelBinarizer()
mlb_directors = MultiLabelBinarizer()
mlb_companies = MultiLabelBinarizer()

vec_genres = mlb_genres.fit_transform(genres_list)
vec_tags = mlb_tags.fit_transform(tags_list)
vec_directors = mlb_directors.fit_transform(directors_list)
vec_companies = mlb_companies.fit_transform(companies_list)

# 전체 특징 벡터 결합
feature_matrix = np.hstack([vec_genres, vec_tags, vec_directors, vec_companies])
print(f"Feature matrix shape: {feature_matrix.shape}")

Feature matrix shape: (8873, 2510)


In [ ]:
# 각 애니메이션별 Top-10 추출
# - 자기 자신 + 동일 그룹(시리즈/자막더빙) 제외
# - 후보군 내에서도 동일 그룹은 대표작 하나만 남김 (similarity 최고 우선, 동점 시 popularity 최고)
TOP_K = 10
similar_rows = []
now = datetime.now()

for i, aid in enumerate(anime_ids):
    my_group = group_ids[i]
    scores = sim_matrix[i]

    # (index, similarity) 리스트 생성 — 자기 그룹 제외
    scored = []
    for j, score in enumerate(scores):
        if i == j:
            continue
        # 동일 그룹(시리즈) 제외
        if my_group is not None and group_ids[j] is not None and my_group == group_ids[j]:
            continue
        if score > 0:
            scored.append((j, float(score)))

    # similarity 내림차순, 동점 시 popularity 내림차순
    scored.sort(key=lambda x: (x[1], popularities[x[0]]), reverse=True)

    # 후보군 내 동일 그룹 중복 제거: 그룹당 대표작 1개만
    seen_groups = set()
    candidates = []
    for j, score in scored:
        candidate_group = group_ids[j]
        if candidate_group is not None:
            if candidate_group in seen_groups:
                continue
            seen_groups.add(candidate_group)
        candidates.append((j, score))
        if len(candidates) >= TOP_K:
            break

    for j, score in candidates:
        similar_rows.append((
            aid,
            anime_ids[j],
            round(score, 4),
            now
        ))

print(f"Total similar pairs: {len(similar_rows)}")

In [ ]:
# anime_similar 테이블 생성 (없는 경우) + 기존 데이터 삭제 후 bulk insert
cur.execute("""
    CREATE TABLE IF NOT EXISTS anime_similar (
        id BIGSERIAL PRIMARY KEY,
        anime_id BIGINT NOT NULL REFERENCES anime(id),
        similar_anime_id BIGINT NOT NULL REFERENCES anime(id),
        similarity NUMERIC(5, 4) NOT NULL,
        created_at TIMESTAMP NOT NULL DEFAULT NOW(),
        UNIQUE(anime_id, similar_anime_id)
    );
""")
cur.execute("TRUNCATE TABLE anime_similar RESTART IDENTITY;")

execute_batch(cur, """
    INSERT INTO anime_similar (anime_id, similar_anime_id, similarity, created_at)
    VALUES (%s, %s, %s, %s);
""", similar_rows, page_size=1000)

pg.commit()
print(f"Inserted {len(similar_rows)} rows")

In [25]:
# bulk insert (ON CONFLICT DO UPDATE)
execute_batch(cur, """
    INSERT INTO anime_similar (anime_id, similar_anime_id, similarity, created_at)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (anime_id, similar_anime_id)
    DO UPDATE SET similarity = EXCLUDED.similarity, created_at = EXCLUDED.created_at;
""", similar_rows, page_size=1000)

pg.commit()
print(f"Inserted/Updated {len(similar_rows)} rows")

Inserted/Updated 86062 rows


In [26]:
# 결과 확인
cur.execute("SELECT COUNT(*) FROM anime_similar")
count = cur.fetchone()[0]
print(f"Total rows in anime_similar: {count}")

# 샘플 확인
cur.execute("""
    SELECT s.anime_id, a1.name, s.similar_anime_id, a2.name, s.similarity
    FROM anime_similar s
    JOIN anime a1 ON s.anime_id = a1.id
    JOIN anime a2 ON s.similar_anime_id = a2.id
    ORDER BY s.anime_id, s.similarity DESC
    LIMIT 10
""")
for row in cur.fetchall():
    print(f"  {row[1]} -> {row[3]} (similarity: {row[4]})")

Total rows in anime_similar: 86062
  라바 시즌 1 -> (자막) 10 라이브즈 (similarity: 0.7071)
  라바 시즌 1 -> (더빙) 바다몬스터 (similarity: 0.7071)
  라바 시즌 1 -> (자막) 바다몬스터 (similarity: 0.7071)
  라바 시즌 1 -> 판다코판다 (similarity: 0.7071)
  라바 시즌 1 -> (더빙) 10 라이브즈 (similarity: 0.7071)
  라바 시즌 1 -> (더빙) 래미의 초특급 시간여행 (similarity: 0.7071)
  라바 시즌 1 -> 돼지저금통TV part 2 (similarity: 0.6804)
  라바 시즌 1 -> 돼지저금통TV part 3 (similarity: 0.6804)
  라바 시즌 1 -> 돼지저금통TV part 1 (similarity: 0.6804)
  라바 시즌 1 -> 스피드광 나무늘보 안드레 (similarity: 0.6804)


In [54]:
cur.close()
pg.close()
print("Done!")

Done!
